In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/archive/data.csv')
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head()

Filas: 9701
Columnas: 67


,cliente_id,genero,edad,pais,ciudad,segmento_cliente,es_mayor,tiene_pareja,tiene_dependientes,latitud,...,dias_mora,cambio_plan_reciente,downgrade_reciente,visitas_app_mensual,tiempo_sesion_promedio,features_nuevas_usadas,ultimo_contacto_soporte,competidores_area,ofertas_recibidas,precio_vs_mercado
0,0002-ORFBO,Femenino,36.0,USA,New York,PYME,0,Si,Si,40.726363,...,0,1,0,66,11.51,3,2025-11-27,4,0,Competitivo
1,0003-MKNFE,Masculino,45.0,USA,New York,Residencial,0,No,No,40.678022,...,0,0,0,51,20.17,6,2025-11-10,1,2,Alto
2,0004-TLHLJ,Masculino,36.0,USA,New York,PYME,0,No,No,40.638134,...,0,0,0,5,109.44,5,2025-10-29,6,2,Competitivo
3,0011-IGKFF,Masculino,NaN,USA,New York,PYME,1,Si,No,40.683321,...,0,0,0,9,90.94,0,2025-11-14,2,3,Competitivo
4,0013-EXCHZ,Femenino,48.0,USA,New York,PYME,1,Si,No,40.764546,...,0,0,0,37,113.28,8,2025-08-12,1,0,Alto


In [2]:
# Eliminar data leakage
cols_eliminar = [
    'nivel_riesgo',
    'score_riesgo',
    'dias_ultima_conexion'
]
df_clean = df.drop(columns=cols_eliminar)

# Corregir typos e inconsistencias
df_clean['respuesta_encuesta'] = df_clean['respuesta_encuesta'].replace('Satifecho', 'Satisfecho')

servicios_cols = ['seguridad_online', 'respaldo_online', 'proteccion_dispositivo',
                  'soporte_tecnico', 'streaming_tv', 'streaming_peliculas']
for col in servicios_cols:
    df_clean[col] = df_clean[col].replace('No internet service', 'Sin servicio internet')

# Imputar nulos
df_clean['edad'] = df_clean['edad'].fillna(df_clean['edad'].median())
df_clean['referencias_hechas'] = df_clean['referencias_hechas'].fillna(0)
df_clean['respuesta_encuesta'] = df_clean['respuesta_encuesta'].fillna('Sin Respuesta')

print(f"Shape: {df_clean.shape}")
print("✅ Limpieza completada")

Shape: (9701, 64)
✅ Limpieza completada


In [3]:
cols_ordenadas = [
    'cliente_id',
    'pais', 'ciudad', 'estado', 'borough', 'codigo_postal', 'latitud', 'longitud',
    'ingreso_mediano', 'densidad_poblacional',
    'genero', 'edad', 'es_mayor', 'tiene_pareja', 'tiene_dependientes', 'segmento_cliente',
    'fecha_registro', 'tipo_contrato', 'antiguedad', 'canal_registro', 'metodo_pago',
    'cargo_mensual', 'ingresos_totales', 'errores_pago', 'descuento_aplicado',
    'aumento_precio_3m', 'facturacion_sin_papel',
    'servicio_telefono', 'lineas_multiples', 'tipo_internet', 'seguridad_online',
    'respaldo_online', 'proteccion_dispositivo', 'soporte_tecnico', 'streaming_tv',
    'streaming_peliculas',
    'conexiones_mensuales', 'dias_activos_semanales', 'promedio_conexion',
    'caracteristicas_usadas', 'tasa_crecimiento_uso', 'tiempo_sesion_promedio',
    'visitas_app_mensual', 'features_nuevas_usadas',
    'tickets_soporte', 'tiempo_resolucion', 'tipo_queja', 'escaladas',
    'ultimo_contacto_soporte',
    'puntuacion_nps', 'puntuacion_csat', 'tasa_apertura_email', 'tasa_clics_marketing',
    'respuesta_encuesta', 'referencias_hechas',
    'fecha_ultimo_pago', 'intentos_cobro_fallidos', 'dias_mora',
    'cambio_plan_reciente', 'downgrade_reciente',
    'competidores_area', 'ofertas_recibidas', 'precio_vs_mercado',
    'cancelacion'
]

df_clean = df_clean[cols_ordenadas]
df_clean.to_csv('../../data/data.csv', index=False)
print(f"✅ Fuente de verdad guardada: {df_clean.shape}")

✅ Fuente de verdad guardada: (9701, 64)


In [5]:
import folium

# Tomar muestra de 500 puntos
sample = df_clean.sample(500, random_state=42)

# Crear mapa centrado en New York
mapa = folium.Map(location=[40.7128, -74.0060], zoom_start=11)

# Agregar puntos
for _, row in sample.iterrows():
    folium.CircleMarker(
        location=[row['latitud'], row['longitud']],
        radius=3,
        color='red',
        fill=True
    ).add_to(mapa)

# Guardar
mapa.save('mapa_verificacion.html')
print("✅ Mapa guardado")
print(f"Lat: {df_clean['latitud'].min():.4f} a {df_clean['latitud'].max():.4f}")
print(f"Lon: {df_clean['longitud'].min():.4f} a {df_clean['longitud'].max():.4f}")

✅ Mapa guardado
Lat: 40.1124 a 40.9102
Lon: -77.5199 a -73.7018
